# US Mortgage Loan Application Prediction: Logistic Regression Baseline

This notebook establishes a rigorous white-box machine learning baseline to predict loan applications (approved vs. rejected) using the Home Mortgage Disclosure Act (HMDA) dataset. It handles data loading, preprocessing pipelines, hyperparameter tuning, and exports evaluation metrics and predictions for fairness and stability audits.

In [ ]:
import polars as pl
import pandas as pd
import joblib
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, classification_report, precision_recall_curve, auc, 
    confusion_matrix, accuracy_score, precision_score, 
    recall_score, f1_score)

DATA_DIR = Path("../data/processed")
TARGET_COL = "target"
RANDOM_STATE = 42

## 1. Data Loading & Stratified Splits
Loads the preprocessed parquet splits (`train`, `val`, `test`), separating the feature space from the target variable (`target`) and protected/sensitive attributes (which are set aside for downstream fairness analysis).

In [ ]:
SENSITIVE_COLS = [
    'applicant_age','census_tract','co_applicant_age',
    'county_code','derived_ethnicity','derived_msa_md',
    'derived_race','derived_sex','state_code','tract_median_age_of_housing_units',
    'tract_minority_population_percent', 'tract_one_to_four_family_homes',
    'tract_owner_occupied_units', 'tract_population', 'tract_to_msa_income_percentage',
    'ffiec_msa_md_median_family_income'
]

def load_and_split_data(split_name):
    df = pl.read_parquet(DATA_DIR / f"{split_name}.parquet")
    feature_cols = [col for col in df.columns if col not in [TARGET_COL] + SENSITIVE_COLS]
    
    X = df.select(feature_cols).to_pandas()
    y = df.select(TARGET_COL).to_pandas().to_numpy().ravel()
    sensitive = df.select(SENSITIVE_COLS).to_pandas()
    
    return X, y, sensitive

X_train, y_train, sensitive_train = load_and_split_data("train")
X_val, y_val, sensitive_val = load_and_split_data("val")
X_test, y_test, sensitive_test = load_and_split_data("test")

## 2. Feature Filtering
Drops columns to prevent overfitting and ensure robust generalization.

In [ ]:
SIMPLICITY_COLS = ["applicant_credit_score_type", "co_applicant_credit_score_type", "aus_1",
"conforming_loan_limit", "construction_method", "derived_loan_product_type", "total_units",
"manufactured_home_secured_property_type", "manufactured_home_land_property_interest", "negative_amortization",
"interest_only_payment", "balloon_payment", "other_nonamortizing_features", "reverse_mortgage",
"hoepa_status", "preapproval", "business_or_commercial_purpose", "initially_payable_to_institution",
"submission_of_application", "open_end_line_of_credit", "discount_points", "lender_credits", "intro_rate_period",
"tract_population", "tract_owner_occupied_units", "tract_one_to_four_family_homes", "tract_median_age_of_housing_units"]


COLUMNS_TO_DROP = ['applicant_age_above_62','applicant_ethnicity_observed','applicant_race_observed',
                   'applicant_sex_observed','co_applicant_age_above_62',
                    'co_applicant_ethnicity_observed','co_applicant_race_observed',
                    'co_applicant_sex_observed','hoepa_status',
                    'preapproval','applicant_ethnicity_1',
                    'applicant_race_1','applicant_sex',
                    'co_applicant_ethnicity_1','co_applicant_race_1',
                    'co_applicant_sex','discount_points',
                    'lender_credits','hoepa_status', 'lei'
                   ] + SIMPLICITY_COLS # Simplicity columns that were removed for the other models

X_train = X_train.drop(columns=COLUMNS_TO_DROP, errors="ignore")
X_val = X_val.drop(columns=COLUMNS_TO_DROP, errors="ignore")
X_test = X_test.drop(columns=COLUMNS_TO_DROP, errors="ignore")

print(f"Features remaining for training ({X_train.shape[1]}):") # Should be 12 columns without the simplicity columns
for col in X_train.columns:
    print(f" - {col}")

## 3. Preprocessing Pipeline Construction
Defines a robust column transformer pipeline:
- **Numeric Features:** Imputes missing values with the median and applies standard scaling.
- **Categorical Features:** Imputes missing values with a constant string ("missing") and applies one-hot encoding with unknown category handling.

In [ ]:
numeric_cols = X_train.select_dtypes(include=["number", "float", "int"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

print("Training model pipeline...")
model_pipeline.fit(X_train, y_train)

In [ ]:
# Output both predictions and predicted probabilities as requested
val_preds = model_pipeline.predict(X_val)
val_probs = model_pipeline.predict_proba(X_val)[:, 1] # Probability of acceptance (class 1)

# Quick validation check
val_auc = roc_auc_score(y_val, val_probs)
print(f"Validation ROC-AUC: {val_auc:.4f}")

## 4. Logistic Regression Hyperparameter Tuning
Performs a grid search over regularization strengths ($C \in \{0.001, 0.01, 0.1, 1.0, 10.0\}$) using the validation set to optimize performance.

In [ ]:
# Transform train and validation sets once using your preprocessor
print("Transforming training data...")
X_train_transformed = preprocessor.fit_transform(X_train)

print("Transforming validation data...")
X_val_transformed = preprocessor.transform(X_val)

# Define a grid of C values to test
c_values = [0.001, 0.01, 0.1, 1.0, 10.0]
best_c = None
best_auc = 0.0
best_model = None

for c in c_values:
    print(f"Training Logistic Regression with C={c}...")
    
    # Initialize model with current C
    # solver='saga' or 'lbfgs' work well for large datasets; 
    # saga is great if you add L1 regularization later.
    lr = LogisticRegression(C=c, max_iter=1000, random_state=42, n_jobs=-1)
    
    # Fit directly on pretransformed data
    lr.fit(X_train_transformed, y_train)
    
    # Evaluate on validation set
    val_probs = lr.predict_proba(X_val_transformed)[:, 1]
    auc = roc_auc_score(y_val, val_probs)
    
    print(f"  -> Validation ROC-AUC: {auc:.5f}")
    
    if auc > best_auc:
        best_auc = auc
        best_c = c
        best_model = lr

print(f"\nBest C found: {best_c} with Validation ROC-AUC: {best_auc:.5f}")

## 5. Final Evaluation & Metric Export
Evaluates the optimal model on the unseen test set, computes comprehensive classification metrics (ROC-AUC, PR-AUC, Precision, Recall, Specificity, F1, and Confusion Matrix), and exports the artifacts to `data/log-reg-results/`.

In [ ]:
# 1. Transform the test data using the fitted preprocessor
print("Transforming test data...")
X_test_transformed = preprocessor.transform(X_test)

# 2. Generate predictions and predicted probabilities
print("Generating test predictions...")
test_probs = best_model.predict_proba(X_test_transformed)[:, 1] # Probability of class 1 (accepted/rejected)
test_preds = best_model.predict(X_test_transformed)

# 3. Final Evaluation
test_auc = roc_auc_score(y_test, test_probs)
print(f"\nFinal Test ROC-AUC: {test_auc:.5f}")

# Optional: Detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, test_preds))

In [ ]:
from sklearn.metrics import auc

In [ ]:
# 1. Ensure the output directory exists
output_dir = Path("../data/log-reg-results")
output_dir.mkdir(parents=True, exist_ok=True)

# 2. Transform test set and generate probabilities
X_test_transformed = preprocessor.transform(X_test)
test_probs = best_model.predict_proba(X_test_transformed)[:, 1]
threshold = 0.5
test_preds = (test_probs >= threshold).astype(int)

# 3. Compute Metrics
test_auc = roc_auc_score(y_test, test_probs)
precision_curve_vals, recall_curve_vals, _ = precision_recall_curve(y_test, test_probs)
test_pr_auc = auc(recall_curve_vals, precision_curve_vals)

accuracy = accuracy_score(y_test, test_preds)
precision_val = precision_score(y_test, test_preds, zero_division=0)
sensitivity_val = recall_score(y_test, test_preds, zero_division=0) # Sensitivity = Recall
f1_val = f1_score(y_test, test_preds, zero_division=0)

# Confusion Matrix breakdown: [[TN, FP], [FN, TP]]
cm = confusion_matrix(y_test, test_preds)
tn, fp, fn, tp = cm.ravel()
specificity_val = tn / (tn + fp) if (tn + fp) > 0 else 0

# Print a quick preview to the notebook output
print("=== Test Set Performance Analysis ===")
print(f"ROC-AUC:       {test_auc:.5f}")
print(f"Accuracy:      {accuracy:.5f}")
print(f"Precision:     {precision_val:.5f}")
print(f"Sensitivity:   {sensitivity_val:.5f}")
print(f"Specificity:   {specificity_val:.5f}")
print(f"F1-Score:      {f1_val:.5f}")
print(f"\nConfusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")

# 4. Create the detailed metrics summary with definitions
metrics_summary = f"""==================================================
LOGISTIC REGRESSION TEST PERFORMANCE SUMMARY
==================================================

1. OVERALL METRICS (Cutoff = {threshold})
--------------------------------------------------
- ROC-AUC:                     {test_auc:.5f}
- PR-AUC:                      {test_pr_auc:.5f}
- Accuracy:                    {accuracy:.5f}
- Precision:                   {precision_val:.5f}
- Sensitivity (Recall / TPR):  {sensitivity_val:.5f}
- Specificity (TNR):           {specificity_val:.5f}
- F1-Score:                    {f1_val:.5f}

2. CONFUSION MATRIX BREAKDOWN
--------------------------------------------------
- True Negatives (TN):         {tn:,}  (Correctly predicted class 0)
- False Positives (FP):        {fp:,}  (Incorrectly predicted class 1 - Type I Error)
- False Negatives (FN):        {fn:,}  (Incorrectly predicted class 0 - Type II Error)
- True Positives (TP):         {tp:,}  (Correctly predicted class 1)

3. DEFINITIONS & INTERPRETATION
--------------------------------------------------
- Probability Cutoff: The threshold (0.5) applied to predicted probabilities to assign binary class labels (>= 0.5 maps to 1, < 0.5 maps to 0).
- ROC-AUC: Area Under the Receiver Operating Characteristic curve. Evaluates the model's capacity to rank positive instances higher than negative ones across all possible thresholds.
- PR-AUC: Area Under the Precision-Recall Curve. Highlights performance focusing specifically on the minority/positive class.
- Accuracy: The proportion of total predictions (both positive and negative) that were correct. Formula: (TP + TN) / Total.
- Precision: Out of all instances predicted as positive (accepted), what fraction were actually positive? Formula: TP / (TP + FP).
- Sensitivity (Recall / True Positive Rate): Out of all actual positive instances, what fraction did the model successfully find? Formula: TP / (TP + FN).
- Specificity (True Negative Rate): Out of all actual negative instances, what fraction did the model correctly identify as negative? Formula: TN / (TN + FP).
- F1-Score: The harmonic mean of precision and sensitivity, balancing both metrics into a single score.
"""

# 5. Save outputs to data/log-reg-results
results_df = sensitive_test.copy()
results_df["true_target"] = y_test
results_df["predicted_probability"] = test_probs
results_df["predicted_label"] = test_preds

results_df.to_parquet(output_dir / "test_predictions_with_sensitive.parquet")
(output_dir / "metrics_summary.txt").write_text(metrics_summary)

print(f"\nResults and detailed metrics successfully stored in: {output_dir.resolve()}")

## 6. Export through the team's shared interface

The cells above already train and tune the model inline. These cells do the same tuning
through `src/logreg_model.py` instead, so the result is usable by the shared
interpretability/stability/fairness notebooks (`notebooks/evaluation/`), which import a
model by its `fit()`/`predict_proba()` functions, not by re-reading this notebook.

`X_train`/`X_val`/`y_train`/`y_val` here are already the 12-column, sensitive-attribute-free
frames from section 2 above — `logreg_model.FEATURES` is exactly that same 12-column set,
derived directly from this notebook's own `SENSITIVE_COLS`/`COLUMNS_TO_DROP` logic.

In [ ]:
import sys
sys.path.insert(0, "../src")
import logreg_model as lm

print(f"logreg_model.FEATURES ({len(lm.FEATURES)}): {lm.FEATURES}")
assert set(lm.FEATURES) == set(X_train.columns), "notebook's trimmed columns and logreg_model.FEATURES disagree"


In [ ]:
print("Tuning C on the validation set via logreg_model.tune()...")
model = lm.tune(X_train, y_train, X_val, y_val)


## 7. Save artifacts

Saves the tuned `C` (so every future `logreg_model.fit(X_train, y_train)` call — including
the stability notebook's bootstrap refits — uses this tuned value instead of the default)
and the fitted model itself.

To load the model elsewhere (SHAP/XPER, stability, fairness notebooks, app), put `src/` on
the path first, same as the xgboost/tabpfn models:

```python
import sys; sys.path.insert(0, "src")
import joblib, logreg_model as lm
model = joblib.load("models/logreg_model.joblib")
p = lm.predict_proba(model, df)[:, 1]
```

In [ ]:
import json

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

lm.PARAMS_PATH.write_text(json.dumps({"C": model.C}, indent=2))
print(f"wrote {lm.PARAMS_PATH} (C={model.C})")

joblib.dump(model, MODELS_DIR / "logreg_model.joblib")
print(f"wrote {MODELS_DIR / 'logreg_model.joblib'}")

test_probs = lm.predict_proba(model, X_test)[:, 1]
test_auc = roc_auc_score(y_test, test_probs)
print(f"test AUC via logreg_model.predict_proba: {test_auc:.5f}")
